# Feature checks

Validates each feature function in `src/sentinel/features/` on the Sparkov data.
Exploration and the reasoning behind each feature are in `01_explore_sparkov.ipynb`.

In [1]:
from pathlib import Path

import pandas as pd
pd.set_option("display.float_format", "{:,.2f}".format)

RAW_DIR = Path("..") / "data" / "raw"

train = pd.read_csv(RAW_DIR / "fraudTrain.csv", index_col=0)
test = pd.read_csv(RAW_DIR / "fraudTest.csv", index_col=0)
df = pd.concat([train, test], ignore_index=True)
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])

print("all:", df.shape)

all: (1852394, 22)


## Feature check: time features (step 6a)

Testing `add_time_features()` from `src/sentinel/features/temporal.py` on the full dataset.

It adds three columns, each using only the transaction's own timestamp, so there is no look-ahead leakage:

| Feature | Meaning |
|---|---|
| `hour` | Hour of day, 0–23 |
| `is_night` | 1 if the hour is 22:00–03:59, the high-risk window found in the hour-of-day chart in 01_explore_sparkov |
| `day_of_week` | 0 = Monday … 6 = Sunday |

**Checks:** `is_night` is 1 only for hours 22–3, and the night fraud rate is far above the day rate.

In [2]:
from sentinel.features.temporal import add_time_features

df_time = add_time_features(df)

print(df_time[["trans_date_trans_time", "hour", "is_night", "day_of_week"]].head())
print()
print(df_time.groupby("is_night")["is_fraud"].agg(["count", "mean"]))

  trans_date_trans_time  hour  is_night  day_of_week
0   2019-01-01 00:00:18     0         1            1
1   2019-01-01 00:00:44     0         1            1
2   2019-01-01 00:00:51     0         1            1
3   2019-01-01 00:01:16     0         1            1
4   2019-01-01 00:03:06     0         1            1

            count  mean
is_night               
0         1417373  0.00
1          435021  0.02


## Feature check: distance from home (step 6b)

Testing `haversine_km()` from `src/sentinel/features/geo.py`.

**First, known answers:** Berlin → Paris should be about 878 km, and a point to itself exactly 0 km.

In [3]:
from sentinel.features.geo import haversine_km

berlin = (52.5200, 13.4050)
paris = (48.8566, 2.3522)

print("Berlin -> Paris: ", round(haversine_km(*berlin, *paris), 1), "km")
print("Berlin -> Berlin:", round(haversine_km(*berlin, *berlin), 1), "km")

Berlin -> Paris:  877.5 km
Berlin -> Berlin: 0.0 km


### Is distance a real signal?

Compares `distance_km` for legitimate (0) and fraudulent (1) transactions, using `add_distance_feature()` from `src/sentinel/features/geo.py`.

| Feature | Meaning |
|---|---|
| `distance_km` | Straight-line distance from the customer's home to the merchant, in km |

**Check:** if fraud's median distance is clearly larger, the feature is useful. If both columns look alike, it is weak on Sparkov.

In [4]:
from sentinel.features.geo import add_distance_feature

df_geo = add_distance_feature(df)

print(df_geo.groupby("is_fraud")["distance_km"].describe().T)

is_fraud            0        1
count    1,842,743.00 9,651.00
mean            76.11    76.26
std             29.12    28.87
min              0.02     0.74
25%             55.32    55.57
50%             78.22    78.10
75%             98.51    98.42
max            152.12   144.52


## Feature check: rolling count features (step 6c)

Testing `add_count_features()` from `src/sentinel/features/behaviour.py` on a single small customer (6 transactions), so every count can be checked by hand.

Each count only includes the customer's **earlier** transactions inside the window, never the current one.

In [5]:
from sentinel.features.behaviour import add_count_features

one_customer = df[df["cc_num"] == 4295296907373]
checked = add_count_features(one_customer)

print(checked[["trans_date_trans_time", "amt", "tx_count_1h", "tx_count_24h", "tx_count_7d"]])

        trans_date_trans_time      amt  tx_count_1h  tx_count_24h  tx_count_7d
1624802   2020-10-25 06:57:18    12.40            0             0            0
1626475   2020-10-25 20:43:20   614.29            0             1            1
1626723   2020-10-25 22:34:45 1,038.53            0             2            2
1626765   2020-10-25 22:51:03   773.18            1             3            3
1626799   2020-10-25 23:02:32   772.55            2             4            4
1626871   2020-10-25 23:32:34 1,061.53            3             5            5
